# 02 · Thin overlapping releases

Rolling polls publish a result almost daily from a sample renewed by a third each day, so consecutive
releases share most of their respondents. Following Erikson and Wlezien, overlap is removed **within a
series** (one pollster working for one client): from the last release backwards, a release is kept only
if its fieldwork ends before the start of the last kept one. Every interview then counts at most once,
and the final pre-election release is always kept.

The client is not known everywhere, so a series is defined as precisely as each source allows:

| source | series |
|---|---|
| france-politique (2002) | the page's table, one per pollster/client (e.g. CSA/Libération) |
| NSPPolls (2022) | pollster + `commanditaire` |
| Wikipedia | only releases the page marks "(rolling)" (2022) |
| any poll in `validation/flagged_clients.csv` with a `client` filled in by hand | pollster + that client |

Wikipedia has no client column, and dates cannot separate a rolling series from two clients polling the
same week (2002: Ifop for the *JDD* on 21–22 Feb and for *L'Express* on 22–23 Feb). Other polls that
overlap a poll by the same pollster are therefore **kept and flagged** `overlap_client_unknown`, so any
analysis can be rerun without them.

Reads `data/interim/polls_*.csv` (notebook 01); writes `data/interim/thinning.csv`, one row per poll.
Nothing is deleted from the poll tables: join on `poll_id` and filter on `kept`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from polls import thinning, worklists

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
INTERIM = ROOT / "data" / "interim"

polls = pd.concat(
    [pd.read_csv(INTERIM / f"polls_{s}.csv") for s in ("wikipedia", "france_politique", "nsppolls")]
).drop_duplicates("poll_id")[thinning.POLL_FIELDS]

In [2]:
CLIENTS = ROOT / "validation" / "flagged_clients.csv"
clients = {}
if CLIENTS.exists():
    c = worklists.read(CLIENTS)
    clients = dict(zip(c["poll_id"], c["client"].str.strip()))
    print(f"Clients looked up by hand: {sum(bool(v) for v in clients.values())} of {len(clients)} polls")

thinned = thinning.thin(polls, clients)
out_cols = ["election", "source", "poll_id", "institute", "series", "fieldwork_start", "fieldwork_end",
            "reported_date", "rolling", "kept", "overlap_client_unknown", "client_from_validation",
            "overlaps_with", "reason"]
thinned[out_cols].sort_values(["election", "source", "poll_id"]).to_csv(INTERIM / "thinning.csv", index=False)

summary = thinned.groupby(["election", "source"]).agg(
    polls=("kept", "size"), kept=("kept", "sum"), flagged_client_unknown=("overlap_client_unknown", "sum"))
summary.insert(2, "dropped", summary["polls"] - summary["kept"])
summary

Clients looked up by hand: 19 of 19 polls


polls  kept  dropped  flagged_client_unknown
election source                                                        
2002     france_politique     98    98        0                       0
         wikipedia            98    98        0                       6
2007     wikipedia           107   107        0                       0
2012     wikipedia           124   124        0                       0
2017     wikipedia           154   148        6                       0
2022     nsppolls            351   230      121                       0
         wikipedia           263   231       32                      25

### Polls whose client was looked up by hand

In [3]:
thinned[thinned["client_from_validation"]][["poll_id", "series", "kept", "overlaps_with", "reason"]]

,poll_id,series,kept,overlaps_with,reason
389,2017_wp_harris-interactive_2017-04-05_1,"harrisinteractive | france television, l’emission politique",True,None,no overlap
390,2017_wp_harris-interactive_2017-04-06_1,harrisinteractive | lcp assemblee nationale,True,None,no overlap
402,2017_wp_ifop-fiducial_2017-02-17_1,"ifop | cnews, paris match, sud radio",True,None,no overlap
403,2017_wp_ifop-fiducial_2017-02-20_1,"ifop | paris match, sud radio",True,None,no overlap
404,2017_wp_ifop-fiducial_2017-02-22_1,"ifop | cnews, paris match, sud radio",True,None,no overlap
406,2017_wp_ifop-fiducial_2017-03-02_1,"ifop | cnews, paris match, sud radio",False,2017_wp_ifop-fiducial_2017-03-04_1,fieldwork overlaps a later kept release of the same series
407,2017_wp_ifop-fiducial_2017-03-04_1,"ifop | cnews, paris match, sud radio",True,None,no overlap
408,2017_wp_ifop-fiducial_2017-03-04_2,"ifop | cnews, paris match, sud radio",False,2017_wp_ifop-fiducial_2017-03-04_1,fieldwork overlaps a later kept release of the same series
409,2017_wp_ifop-fiducial_2017-03-07_1,"ifop | cnews, paris match, sud radio",False,2017_wp_ifop-fiducial_2017-03-10_1,fieldwork overlaps a later kept release of the same series
410,2017_wp_ifop-fiducial_2017-03-10_1,"ifop | cnews, paris match, sud radio",True,None,no overlap


### Dropped releases, by series

In [4]:
thinned[~thinned["kept"]].groupby(["election", "source", "series"]).size().rename("dropped").to_frame()

dropped
election source    series                                                        
2017     wikipedia ifop | cnews, paris match, sud radio                         5
                   opinionway | les echos, radio classique                      1
2022     nsppolls  Ifop | lci, paris match, sud radio                          44
                   Ipsos | france info, le parisien                            17
                   OpinionWay | cnews                                          11
                   OpinionWay | kea partners, les echos, radio classique       49
         wikipedia ifop [rolling]                                              13
                   ipsos [rolling]                                              4
                   opinionway [rolling]                                        14
                   opinionway | cnews                                           1

### Kept polls flagged `overlap_client_unknown`

Wikipedia polls that overlap another poll by the same pollster, outside any rolling-marked series. In 2002
the france-politique tables show these are separate clients; in 2017 and 2022 they may be unmarked rolling
releases. The 2017 Wikipedia rows often link the pollster's report, which names the client.

In [5]:
thinned[thinned["overlap_client_unknown"]].groupby(["election", "institute"]).size().rename("flagged").to_frame()

flagged
election institute          
2002     Ifop              4
         Sofres            2
2022     Ipsos             4
         OpinionWay       21

### Diagnostic: overlaps between different series of the same pollster

Kept polls by one pollster, for different clients, whose fieldwork overlaps. Usually independent polls; a
series split by an inconsistent client name would also appear here. In 2022, OpinionWay's CNews releases
overlap its *Les Echos* rolling releases, which may mean one rolling sample sold to two clients: worth
checking against the Commission des sondages notices.

In [6]:
resid = thinning.residual_overlaps(thinned)
print(f"{len(resid)} overlapping pairs")
resid.groupby(["election", "source", "institute", "series_a", "series_b"]).size().rename("pairs").to_frame()

60 overlapping pairs


pairs
election source           institute          series_a                                                    series_b                                                    
2002     france_politique Ifop               Ifop | IFOP/Journal du Dimanche                             Ifop | IFOP/L'Express                                      2
                          Sofres             Sofres | SOFRES/LCI/Politique Opinion                       Sofres | SOFRES/Nouvel Observateur                         1
                                             Sofres | SOFRES/Nouvel Observateur                          Sofres | SOFRES/LCI/Politique Opinion                      1
2017     wikipedia        Harris Interactive harrisinteractive | france television, l’emission politique harrisinteractive | lcp assemblee nationale                1
                          Ifop               ifop | cnews, paris match, sud radio                        ifop | paris match, sud radio                              1
                                             ifop | paris match, sud radio                               ifop | cnews, paris match, sud radio                       1
2022     nsppolls         Ifop               Ifop | lci, paris match, sud radio                          Ifop | journal du dimanche                                 1
                                             Ifop | le jdd                                               Ifop | lci, paris match, sud radio                         1
                          Ipsos              Ipsos | cevipof, fondation jean jaures, le monde            Ipsos | france info, le parisien                           2
                                             Ipsos | france info, le parisien                            Ipsos | cevipof, fondation jean jaures, le monde           4
                                                                                                         Ipsos | le monde                                           1
                          OpinionWay         OpinionWay | cnews                                          OpinionWay | kea partners, les echos, radio classique      5
                                             OpinionWay | kea partners, les echos, radio classique       OpinionWay | cnews                                        15